# Optimizer sandbox

Play with a real `DataPackage` outside of Frappe/bench: tweak employees, FTE targets, rules and weights, re-solve, and inspect the result. No live site needed once you have a snapshot.

**1. Capture a snapshot** from an existing Optimizer Run on `development.localhost` (any status — only its `date`/`mode`/`ruleset`/leave-speculations/existing-assignments mode are used):

```bash
bench --site development.localhost capture-datapackage --run <run-name>
```

This writes `sandbox/snapshots/<run-name>.json`. Snapshots contain real employee/leave data from your dev site, so `sandbox/snapshots/` is gitignored — re-capture instead of committing one.

**2. Pick a kernel**: the app's own Python env (`uv run python` from `apps/autoshift`, or the bench env — both already have `pulp`, `pandas`, and `autoshift` importable; see `pyproject.toml`'s `dev` dependency group for `ipykernel`/`pandas`).

In [ ]:
import logging
from pathlib import Path

import pulp
from helpers import *

from autoshift.optimizer.rules import BUILTIN_RULES
from autoshift.optimizer.types import DataPackage

logging.basicConfig(level=logging.DEBUG)
logging.getLogger().setLevel(logging.DEBUG)

## Load a snapshot

Point this at whichever file `capture-datapackage` produced.

In [ ]:
SNAPSHOT = Path("snapshots/AS-2026-06-22-052.json")

data = DataPackage.loads(SNAPSHOT.read_text())
active_rule_count = len(data.rules) or len(BUILTIN_RULES)
print(
	f"{len(data.employees)} employees, {len(data.shift_types)} shift types, "
	f"{len(data.working_days)} days ({data.working_days[0]}..{data.working_days[-1]}), "
	f"{len(data.branches)} branches, {active_rule_count} rules"
)

## Solve as captured

In [ ]:
prob, x, active_rooms, logs = solve(data)
print(status(prob), pulp.value(prob.objective))

In [ ]:
prob.constraints()[0].name

## Inspect the solution

In [ ]:
ass = assignment_frame(data, x)
ass

In [ ]:
constraint_frame(prob)

In [ ]:
ass[ass["employee"] == "113"]

In [ ]:
room_utilization_frame(data, active_rooms)

In [ ]:
# per-rule contribution to the objective (built-in objective rules only — see helpers.py)
objective_breakdown(data, x, active_rooms)

## Schedule grid

Simplified, offline view of the solved schedule (employee x day), analogous to
`OptimizerRun.get_schedule_events` but without the Frappe/DB-backed "existing" overlay.

In [ ]:
from helpers import schedule_grid  # ty:ignore[unresolved-import]

schedule_grid(data, x)

## Play with variables

`DataPackage` is frozen — use `replace(data, field=...)` (shorthand for `dataclasses.replace`) to build a variant, then re-solve and compare.

In [ ]:
variant = data  # start here, then override fields above
prob2, x2, active_rooms2, _ = built = solve(variant, solve=False)

In [ ]:
for var in prob2.variables():
	var.cat = pulp.LpContinuous

In [ ]:
print(status(prob2), pulp.value(prob2.objective))
prob2, x2, active_rooms2, _ = built = solve(variant, solve=built)
print(status(prob2), pulp.value(prob2.objective))
assignment_frame(variant, x2)

In [ ]:
(a := constraint_frame(prob2))

In [ ]:
a.nunique()

In [ ]:
a["slack"].value_counts()

In [ ]:
a["type"].value_counts()

In [ ]:
a["len"].value_counts().sort_index()

In [ ]:
a[
	[
		"type",
		"slack",
		"pi",
	]
].value_counts().sort_index()

In [ ]:
a.drop_duplicates(subset=["type", "pi", "slack"])

In [ ]:
variant3 = data  # start here, then override fields above
prob3, x3, active_rooms3, _ = built = solve(variant, solve=False)

In [ ]:
prob3.constraints()[0].name = "thefirstrule||:"

print(status(prob3), pulp.value(prob3.objective))
prob3, x3, active_rooms3, _ = built = solve(variant, solve=built)
print(status(prob3), pulp.value(prob3.objective))
# assignment_frame(variant, x3)
(a := constraint_frame(prob3))
# a[['type', 'pi', 'slack']].value_counts().sort_index()
a.drop_duplicates(subset=["type", "pi", "slack"])

In [ ]:
# example: give one employee a bigger FTE target
# employee = data.employees[0]
# variant = replace(data, target_shifts={**data.target_shifts, employee: 40})

variant = data  # start here, then override fields above
prob4, x4, active_rooms4, _ = solve(variant)
print(status(prob4), pulp.value(prob4.objective))
assignment_frame(variant, x4)

## Tweak rule weights / selection

`data.rules` is a tuple of `(rule_name, builtin_key, custom_code, weight)`. An empty tuple means "every built-in rule at weight 1.0". Rebuild it to change weights, drop constraint rules (careful — most exist for correctness, not just "nice to have"), or add a rule you're drafting in `autoshift/optimizer/rule_scratchpad.py`.

In [ ]:
# # example: double the room-utilization objective's weight
# base_rules = data.rules or tuple((rule.title, k, "", 1.0) for k, rule in BUILTIN_RULES.items())
# reweighted = replace(
# 	data,
# 	rules=tuple(
# 		(name, key, code, weight * 2 if key == "room_utilization_objective" else weight)
# 		for name, key, code, weight in base_rules
# 	),
# )
# prob3, x3, active_rooms3, _ = solve(reweighted)
# objective_breakdown(reweighted, x3, active_rooms3)

In [ ]:
# example: double the room-utilization objective's weight
import itertools

from autoshift.optimizer.rules import RuleContext

base_rules = data.rules or tuple((rule.title, k, "", 1.0) for k, rule in BUILTIN_RULES.items())


def weigh_assignments_objective(ctx: RuleContext) -> None:
	data = ctx.data
	epsilon = 2**-10
	ctx.add_objective(
		pulp.lpSum((0 if comb in data.forced else -epsilon) * var for comb, var in ctx.x.items())
	)


new_rule = (
	"Objective: Conserve Existing Assignments",
	"",
	"""
def apply(ctx: RuleContext) -> None:
	data = ctx.data
	epsilon = 2**-10
	ctx.add_objective(
		pulp.lpSum((0 if comb in data.forced else -epsilon) * var for comb, var in ctx.x.items())
	)
	""",
	1.0,
)
reweighted = replace(
	data,
	rules=[*base_rules, new_rule],
)
prob3, x3, active_rooms3, _ = solve(reweighted)
objective_breakdown(reweighted, x3, active_rooms3)

In [ ]:
df = assignment_frame(reweighted, x3)

In [ ]:
df.nunique()

In [ ]:
p = pulp.LpProblem("helelo", pulp.LpMaximize)
x = p.add_variable("x", 0, 5)
y = p.add_variable("y", 0, 10)
p += x + y
# p += x <= 5  # upbound
# p += y <= 10  # upbound
p += x + y <= 12
p.solve(pulp.PULP_CBC_CMD(msg=False))

print(f"Status: {pulp.LpStatus[p.status]}")
print(f"Optimal x: {pulp.value(x)}, y: {pulp.value(y)}")
print(f"Objective: {pulp.value(p.objective)}")
for constraint_name, constraint in p.constraints.items():
	pi = constraint.pi  # Dual value / shadow price
	print(f"{constraint_name}: slack={constraint.slack:.2f}, pi={pi}")
import math

obj_value = pulp.value(p.objective)

p += p.objective == obj_value - 2e-20

solutions = [
	{
		v.name: [
			v.varValue,
			v.dj,
			v.varValue - v.upBound,
		]
		for v in p.variables()
	}
]
while not next(
	(
		setattr(v, "upBound", v.upBound - 0.125)
		for v in p.variables()
		if math.isclose(v.dj, (v.varValue - v.upBound), abs_tol=2e-20)
		and math.isclose((v.varValue - v.upBound), 0.0, abs_tol=2e-20)
	),
	True,
):
	p.solve()
	solutions.append(
		{
			**{
				v.name: [
					v.varValue,
					v.dj,
					v.varValue - v.upBound,
				]
				for v in p.variables()
			},
			**{
				f"c_{c.name}": [
					c.pi,
					c.slack,
				]
				for c in p.constraints()
			},
		}
	)
	if p.status != pulp.LpStatusOptimal:
		print(f"not opti {p.status=}")
		break
	if pulp.value(p.objective) < obj_value:
		print(f"objectif lune {pulp.value(p.objective)} < {obj_value}")
		break
solutions

In [ ]:
{
	v.name: [
		v.varValue,
		v.dj,
		v.varValue - v.upBound,
	]
	for v in p.variables()
}

In [ ]:
p